# Modulo de Video - Analise de fisioterapia (postura / marcha)

Recebe um video de fisioterapia e produz um alerta padronizado de risco (JSON), a partir da analise de postura e movimento do corpo.

> Ambiente: em *Ambiente de execucao > Alterar o tipo de ambiente*, **CPU ja basta**. GPU nao e necessaria.

## Como usar este notebook

Rode as celulas na ordem, de cima para baixo. Sao duas partes:

1. **Pipeline** (passos 1 a 9): prepara o ambiente e roda a analise em um video, gerando o alerta e o video anotado.
2. **Calibracao** (passos 10 a 16): usa o dataset publico REHAB24-6 para ajustar, com dados reais, os limiares que definem o que e "anomalo", e compara o efeito no resultado.

Cada passo tem um texto curto explicando o que faz, seguido da celula que executa.

## 1. Verificar o ambiente

Confere a versao do Python e se ha GPU (nao precisa).

In [ ]:
import sys, platform
print('Python:', sys.version)
print('SO:', platform.platform())
!nvidia-smi -L 2>/dev/null || echo 'Sem GPU (tudo bem, o pipeline roda em CPU).'

## 2. Clonar o repositorio

Clona o repositorio do projeto e entra na pasta `modulo_video`. Se ja estiver clonado, apenas atualiza (`git pull`).

In [ ]:
import os
PROJETO = '/content/projeto'
MODULO = PROJETO + '/modulo_video'
if not os.path.exists(PROJETO):
    !git clone https://github.com/elbergaliza/Tech-Challenge-FIAP-FASE4.git {PROJETO}
else:
    !cd {PROJETO} && git pull
%cd {MODULO}

## 3. Instalar as dependencias

Instala MediaPipe (pose), OpenCV (video) e Ultralytics (YOLOv8). Leva 1 a 2 min.

> Os avisos vermelhos de `ERROR: pip's dependency resolver...` sao **esperados** e inofensivos: o Colab ja vem com bibliotecas que pedem versoes diferentes de numpy/protobuf, mas nenhuma delas e usada por este modulo. Se a celula terminar com "Dependencias instaladas.", esta tudo certo.

In [ ]:
!pip install -q -r requirements.txt
# No Colab (Python 3.12) o mediapipe tenta importar o tensorflow pre-instalado e
# quebra por conflito de protobuf. Como este modulo NAO usa tensorflow, removemos
# para o mediapipe carregar normalmente (a pose estimation nao precisa dele).
!pip uninstall -y tensorflow tensorflow-cpu 2>/dev/null
print('Dependencias instaladas.')

## 4. Como funciona o pipeline (visao geral)

O modulo liga 5 etapas em sequencia, mais uma etapa de objetos:

```
video --(1) MediaPipe Pose --> pontos do corpo (postura)
      --(1b) YOLOv8 --> pessoas/eventos (queda, ausencia, contagem)
                |
                v
   (2) biomecanica --> angulos, simetria, estabilidade, velocidade
   (3) regras clinicas --> anomalias
   (4) score de risco --> 0 a 1 + nivel (baixo/moderado/alto)
   (5) relatorio --> alerta JSON padronizado
```

**Modelos usados:**
- **MediaPipe Pose**: estima os pontos do corpo (keypoints/landmarks) em cada frame, base para a analise de postura e marcha.
- **YOLOv8**: detecta pessoas no quadro, usado para eventos de seguranca (possivel queda, contagem de pessoas, paciente ausente).

**Por que regras e nao um modelo treinado:** os modelos de visao ja sao pre-treinados; sobre os landmarks aplicamos regras simples (limiares) que caracterizam cada desvio. Esses limiares sao calibrados na parte 2 deste notebook.

## 5. Enviar um video

Faca upload de um `.mp4` com a pessoa de **corpo inteiro** visivel (do ombro ao tornozelo) e, de preferencia, uma pessoa so em foco. O arquivo vai para `data/entrada/`.

**Qual video usar:** um agachamento do REHAB24-6, pasta `Ex6`, arquivo `Camera17` (para casar com os limiares calibrados no passo 13). Extraia um do `videos.zip`, por exemplo `PM_029-Camera17-30fps.mp4`.

- Agachamentos disponiveis (Ex6): PM_008, PM_022, PM_029, PM_038, PM_043, PM_105, PM_113, PM_118, PM_126.
- Para ver a diferenca antes/depois no passo 15, prefira um com repeticoes incorretas, como **PM_029** (10 corretas / 10 incorretas). Um quase todo correto, como PM_038, tende a sair "normal".

In [ ]:
from google.colab import files
import shutil, os
os.makedirs('data/entrada', exist_ok=True)
enviados = files.upload()
nome_video = list(enviados.keys())[0]
destino = os.path.join('data/entrada', nome_video)
shutil.move(nome_video, destino)
print('Video salvo em:', destino)

## 6. Rodar o pipeline

Gera o **alerta JSON** e o **video anotado** com o esqueleto desenhado.

> Se o video for pesado e demorar, rode a mesma linha com `--sem-objetos` para desligar o YOLOv8.

In [ ]:
PATIENT_ID = 'video_001'
import os
os.makedirs('data/saida', exist_ok=True)
saida_json = 'data/saida/' + PATIENT_ID + '.json'
saida_anotado = 'data/saida/' + PATIENT_ID + '_anotado.mp4'
!python main.py --video "{destino}" --patient-id {PATIENT_ID} --saida "{saida_json}" --anotado "{saida_anotado}"


## 7. Ver o alerta gerado

Este e o produto final do modulo: o alerta no schema padronizado (detalhado no passo 15).

In [ ]:
import json
with open(saida_json, encoding='utf-8') as f:
    alerta = json.load(f)
print(json.dumps(alerta, ensure_ascii=False, indent=2))

## 8. Visualizar o video anotado

Mostra o esqueleto desenhado sobre o video (funciona melhor com clipes curtos). Para videos grandes, use o passo 9.

In [ ]:
import os, subprocess
from IPython.display import HTML, display
from base64 import b64encode

if not (os.path.exists(saida_anotado) and os.path.getsize(saida_anotado) > 0):
    print('Video anotado nao foi gerado (verifique o passo 6).')
else:
    # O OpenCV grava em mp4v, que o navegador nao toca inline. Reencoda para
    # H264 (libx264) com o ffmpeg do Colab para exibir dentro do notebook.
    saida_web = saida_anotado.replace('.mp4', '_web.mp4')
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', saida_anotado,
                    '-vcodec', 'libx264', '-pix_fmt', 'yuv420p', saida_web], check=False)
    alvo = saida_web if (os.path.exists(saida_web) and os.path.getsize(saida_web) > 0) else saida_anotado
    dados = open(alvo, 'rb').read()
    if len(dados) > 40_000_000:
        print('Video grande demais para exibir inline. Use o passo 9 para baixar.')
    else:
        url = 'data:video/mp4;base64,' + b64encode(dados).decode()
        display(HTML('<video width=480 controls><source src="' + url + '" type="video/mp4"></video>'))

## 9. Baixar os resultados

Baixa o alerta JSON e o video anotado gerados.

In [ ]:
from google.colab import files
import os
files.download(saida_json)
if os.path.exists(saida_anotado) and os.path.getsize(saida_anotado) > 0:
    files.download(saida_anotado)

## 10. Calibracao dos limiares com o REHAB24-6

Ate aqui os limiares de "anomalia" eram chutes iniciais. Agora usamos um dataset **publico** para ajusta-los com dados reais.

**Dataset: [REHAB24-6](https://zenodo.org/records/13305826)** (Zenodo, video RGB, download livre, licenca CC-BY-NC). Cada repeticao de exercicio tem rotulo **execucao correta (1) x incorreta (0)**, o que mapeia direto no conceito de anomalia (movimento fora do padrao esperado).

**Isto e calibracao, nao treinamento.** Nao treinamos modelo nenhum: rodamos o mesmo pipeline nas execucoes corretas e incorretas e comparamos as distribuicoes das metricas para escolher onde cortar o limiar.

> Por que nao o KIMORE: o RGB do KIMORE so e liberado sob pedido ao autor + EULA; o download publico dele traz so profundidade e esqueleto, incompativel com o MediaPipe.

## 11. Conectar o Google Drive e preparar o dataset

Baixe do Zenodo apenas **`videos.zip`** e **`Segmentation.csv`** e coloque numa pasta `REHAB24-6` no seu Drive (pode ignorar os zips de joints/markers). A celula abaixo monta o Drive, copia o CSV e descompacta os videos numa copia local (mais rapida que ler do Drive).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil
DRIVE = '/content/drive/MyDrive/REHAB24-6'   # ajuste se deu outro nome a pasta
REHAB = '/content/REHAB24-6'
os.makedirs(REHAB + '/videos', exist_ok=True)

assert os.path.exists(DRIVE + '/Segmentation.csv'), 'Nao achei Segmentation.csv na pasta do Drive.'
shutil.copy(DRIVE + '/Segmentation.csv', REHAB + '/Segmentation.csv')

if not os.listdir(REHAB + '/videos'):
    print('Descompactando videos.zip (1 a 2 min)...')
    with zipfile.ZipFile(DRIVE + '/videos.zip') as z:
        z.extractall(REHAB + '/videos')
print('OK. Itens na pasta de videos:', len(os.listdir(REHAB + '/videos')))

## 12. Validar a estrutura (dry-run)

Nao processa nada: so confere que cada gravacao do `Segmentation.csv` casou com um arquivo de video. Usamos `--camera Camera17` (a camera de referencia das anotacoes) para pegar uma camera so. O "ambiguo" deve ficar em 0.

In [ ]:
%cd /content/projeto/modulo_video
!python calibrar.py --raiz /content/REHAB24-6 --listar --camera Camera17

## 13. Calibrar um exercicio

Processa o exercicio 6 (**agachamento**), que envolve o corpo todo e faz as metricas de tronco e estabilidade aparecerem bem (exercicios so de braco mostrariam pouco). No fim imprime o comparativo **correto x incorreto** e sugere os limiares para o `config.py`.

O `--salvar-limiares` grava os valores calibrados num arquivo (usado no passo 15). O `config.py` nao e alterado.

> Para um teste rapido, acrescente `--max 4` (processa so 4 gravacoes). Sem ele, roda todas as gravacoes do exercicio (uns minutos por video). O MediaPipe fica silencioso enquanto processa, entao parece parado.

In [ ]:
!python calibrar.py --raiz /content/REHAB24-6 --camera Camera17 --exercise 6 --pular-frames 5 --salvar-limiares data/saida/limiares.json

## 14. Comparar os limiares antes x depois da calibracao

Le o `calibracao.csv` (gerado no passo 13) e mede, para cada metrica, o quanto o limiar **antes** (valor atual do `config.py`) e o limiar **depois** (calibrado a partir dos dados) separam execucao correta de incorreta.

Para cada limiar mostra a **sensibilidade** (% das execucoes incorretas que ele pega) e o **falso positivo** (% das corretas que ele acusa por engano). Um bom limiar tem sensibilidade alta e falso positivo baixo.

In [ ]:
!python avaliar.py --exercise 6

## 15. Comparar antes x depois em um video

Roda o video enviado no passo 5 e mostra dois alertas sobre as **mesmas metricas**: com os limiares do `config.py` (antes) e com os calibrados do passo 13 (depois). O `config.py` nao muda; os calibrados sao sobrepostos so nesta execucao.

> Faz mais sentido se o video do passo 5 for do mesmo tipo de exercicio calibrado (agachamento). Acrescente `--sem-objetos` para rodar mais rapido (so a postura).

In [ ]:
!python comparar_video.py --video "{destino}" --limiares data/saida/limiares.json

## 16. (opcional) Baixar o CSV de calibracao

O `calibracao.csv` tem as metricas por repeticao, com o rotulo correto/incorreto.

In [ ]:
from google.colab import files
import os
if os.path.exists('data/saida/calibracao.csv'):
    files.download('data/saida/calibracao.csv')
else:
    print('Rode o passo 13 primeiro para gerar o calibracao.csv')

## 17. Saida padronizada

O alerta segue um schema fixo:

```json
{
  "patient_id": "...",
  "modulo": "video_fisioterapia",
  "tipo_anomalia": "movimento",
  "score_risco": 0.0,
  "nivel_risco": "baixo|moderado|alto",
  "descricao": "...",
  "recomendacao": "..."
}
```

O `score_risco` (0 a 1) e o `nivel_risco` resumem o risco detectado; a `descricao` lista as anomalias encontradas e a `recomendacao` sugere a acao. O `patient_id` identifica o paciente.

## Observacoes

- **CPU basta**: o gargalo (MediaPipe + YOLOv8) roda em CPU. GPU acelera pouco aqui.
- **Codec**: no Colab (Linux) o video anotado sai em H264, que toca inline no notebook. O codigo escolhe o codec automaticamente por sistema.
- **Conflito mediapipe x tensorflow (Colab)**: se aparecer `cannot import name 'runtime_version'`, va em *Ambiente de execucao > Reiniciar sessao* e rode de novo a partir do passo 2.
- **Desempenho**: para acelerar, use videos curtos/720p, ou desative o YOLOv8 com `--sem-objetos`.